# CartPole: build it, control it, then let it learn

**A type-it-yourself notebook for basic Python learners.**

You will build a small cart-and-pole simulator, discover why velocity matters, build a controller, and train a Q-learning agent. You do not need prior reinforcement learning, economics, or game theory.

Instructional code lives in Markdown. The empty cell underneath is yours to type into. Challenge cells have no displayed answer; the next cell contains runnable tests. Solutions are at the end—scroll there only after trying.

This notebook uses only Python's standard library: `math` and `random`. No installation, NumPy, Gymnasium, Pygame, network access, files, or native extensions are required. It targets Pyodide's Python runtime. Optional browser display uses `js` and `IPython` only if your notebook host provides them; the text view works without either.

Run cells in order. After editing a function, rerun its definition and the cells that use it. Restarting the kernel clears every variable. Empty cells are intentional: 'Run all' does not complete the lesson for you.

This is a self-contained educational CartPole implementation, not an exact Gymnasium environment. Physics uses coupled cart/pole equations with a simple numerical time step. Learning outcomes vary; a failed learning run is evidence to investigate, not proof that you typed incorrectly.

### Your route
1. Read the state and see the pole.
2. Build one physics step and an episode.
3. Discover the limitations of angle-only control.
4. Turn continuous observations into table addresses.
5. Calculate and implement Q-learning.
6. Compare trained, random, and hand-written controllers on the same starts.

Keep a written prediction before each discovery. An episode is one attempt from reset until failure or a time limit. An action is one decision during that attempt.


## 1. Four measurements tell us what is happening

A state is a snapshot represented by a tuple: `(x, velocity, angle, angular_velocity)`. `x` is cart position in metres; velocity is metres per second. Angle is radians from upright, positive when leaning right; angular velocity is radians per second, positive when rotating right. A radian measures angle using arc length divided by radius; π radians is half a turn. Thus 12 degrees is `12 * pi / 180` radians.

`import math` gives access to named mathematical functions. `from ... import ...` brings only the requested name into scope. `Random(7)` creates a separate seeded random-number generator: repeating the same seed repeats its sequence. `uniform(a, b)` chooses a real number between the bounds. A tuple groups values in order. Unpacking assigns its four entries to four names.

Type this into the empty cell below:

```python
import math
from random import Random
rng = Random(7)
state = (0.0, 0.0, 0.04, 0.0)
x, velocity, angle, angular_velocity = state
print("angle in degrees:", angle * 180 / math.pi)
```

### Challenge: Recognize failure

Write `failed(state)`. Return `True` if the cart is strictly outside ±2.4 metres OR the pole is strictly outside ±12 degrees. Equality is still allowed. `abs(number)` removes its sign. `or` is true when either comparison is true. `return` hands a result to the caller.

Hint: unpack the tuple and compare `abs(x)` and `abs(angle)` with their limits.

Write your answer in the empty cell. Run the following test cell afterward. An assertion failure means the stated requirement has not been met; a NameError usually means you need to run or complete an earlier cell.

In [ ]:
assert failed((2.41, 0, 0, 0)), "An escaped cart must fail."
assert failed((0, 0, -0.22, 0)), "A large negative angle must fail too."
assert not failed((2.4, 10, 12 * math.pi / 180, 10)), "Use strict limits; speeds alone do not cause failure."
assert not failed((0, 0, 0, 0))
print("Failure checks passed.")

## 2. See a state before learning anything

A text view works in any Python notebook. `round` converts a coordinate to its nearest integer. `min` and `max` keep it inside the display. A list holds editable characters; indexing selects a position. `join` combines the characters into one string. The pole symbol shows lean, while the printed numbers preserve the full state.

Type this into the empty cell below:

```python
def show(state):
    x, velocity, angle, angular_velocity = state
    track = ["."] * 49
    column = round((x + 2.4) / 4.8 * 48)
    column = min(48, max(0, column))
    track[column] = "C"
    pole = "/" if angle > 0.01 else "\\" if angle < -0.01 else "|"
    print("".join(track), "pole:", pole)
    print("x, speed, angle, rotation:", tuple(round(v, 3) for v in state))
show(state)
```

### Discovery: same position, different futures

Type two states with angle `0.04`, one with angular velocity `-0.8`, another with `0.8`. Call `show` on each. Which pole is already returning toward upright? Can an angle alone distinguish them? The text symbol cannot; the fourth measurement can.


## 3. Make one physics step

Action `0` pushes left; action `1` pushes right. A force changes acceleration, acceleration changes velocity, and velocity changes position. A 0.02-second step means 50 decisions per simulated second.

Here `g` is gravity, `mc` cart mass, `mp` pole mass, and `length` the distance to the pole's centre of mass (half its full length). `sin(angle)` measures sideways lean; `cos(angle)` projects directions. `omega ** 2` means omega squared. Pole rotation and cart motion affect each other, so we calculate both accelerations before updating the state.

The coupled equations below are a provided physical model: you are learning how its quantities enter the program, not being asked to derive rigid-body mechanics. `temp` collects horizontal force and the rotational contribution, divided by total mass. `alpha` is angular acceleration. `acceleration` is horizontal acceleration. The `4 / 3` factor accounts for the rotational inertia of a uniform pole about its pivot. For a different physical pole these equations would change.

Each update uses the OLD velocity: `new position = old position + old velocity * dt`. This is explicit Euler integration, an approximation that accumulates error. `raise ValueError` stops invalid actions instead of silently treating them as valid.

Type this into the empty cell below:

```python
def step(state, action):
    if action not in (0, 1):
        raise ValueError("action must be 0 (left) or 1 (right)")
    x, velocity, angle, omega = state
    g, mc, mp, length, dt = 9.8, 1.0, 0.1, 0.5, 0.02
    force = -10.0 if action == 0 else 10.0
    total_mass = mc + mp
    sine, cosine = math.sin(angle), math.cos(angle)
    temp = (force + mp * length * omega ** 2 * sine) / total_mass
    alpha = (g * sine - cosine * temp) / (length * (4 / 3 - mp * cosine ** 2 / total_mass))
    acceleration = temp - mp * length * alpha * cosine / total_mass
    return (x + velocity * dt,
            velocity + acceleration * dt,
            angle + omega * dt,
            omega + alpha * dt)
print(step((0, 0, 0, 0), 1))
```

In [ ]:
right = step((0, 0, 0, 0), 1)
left = step((0, 0, 0, 0), 0)
assert right[1] > 0 and right[3] < 0, "Moving the base right initially rotates the pole left."
assert all(math.isclose(a, -b, abs_tol=1e-12) for a, b in zip(right, left))
print("Physics direction and symmetry checks passed.")

### Discovery: why pushing right can help a right-leaning pole

Predict the sign of angular acceleration when the pole leans right and the cart is pushed right. Compare `step((0, 0, 0.04, 0), 0)` and the same call with action `1`. Look at the fourth entry. The pole does not instantly change angle: the push first changes rotation speed. Move the base underneath the falling pole.


## 4. Run a whole attempt

`reset` generates four small starting measurements. The tuple comprehension repeats the expression four times. A policy is a function that receives a state and returns an action. Passing a function into another function lets us reuse the same experiment with different decision rules.

A list stores the history. `append` adds a state at the end. `range(limit)` produces exactly `limit` loop iterations. `break` exits early. `len(history) - 1` counts transitions because the initial state was stored before the first transition. Our score counts every attempted step, including a final failing step. A time limit ends evaluation even if the pole remains balanced.

Type this into the empty cell below:

```python
def reset(generator):
    return tuple(generator.uniform(-0.05, 0.05) for _ in range(4))

def episode(policy, start, limit=500):
    current = start
    history = [current]
    for _ in range(limit):
        current = step(current, policy(current))
        history.append(current)
        if failed(current):
            break
    return len(history) - 1, history

def random_policy(state):
    return rng.randrange(2)

score, history = episode(random_policy, reset(rng))
print("steps:", score, "seconds:", score * 0.02)
show(history[-1])
```

### Challenge: Write an angle-only controller

Define `angle_policy(state)` that pushes right when angle is positive, otherwise left. No training yet: you are writing the rule yourself. The goal is to establish a baseline, not to claim that this rule balances well.

Write your answer in the empty cell. Run the following test cell afterward. An assertion failure means the stated requirement has not been met; a NameError usually means you need to run or complete an earlier cell.

In [ ]:
assert angle_policy((0, 0, 0.1, 0)) == 1
assert angle_policy((0, 0, -0.1, 0)) == 0
assert angle_policy((0, 0, 0, 0)) == 0
print("Angle controller checks passed.")

## 5. React to where the pole is heading

We can combine present lean and rotation speed: `angle + 0.15 * omega`. This roughly anticipates where the angle may be 0.15 seconds later if its current rotation continues. It is a heuristic, not an exact forecast. Try different coefficients. This controller has no memory and learns nothing.

Type this into the empty cell below:

```python
def predictive_policy(state):
    x, velocity, angle, omega = state
    prediction = angle + 0.15 * omega
    return 1 if prediction > 0 else 0

start = (0, 0, 0.04, -0.8)
print("angle-only:", angle_policy(start), "predictive:", predictive_policy(start))
for policy in (angle_policy, predictive_policy):
    score, history = episode(policy, start)
    print(policy.__name__, score)
    for snapshot in history[::max(1, len(history) // 5)]:
        show(snapshot)
```

### Discovery: balancing the pole is not the whole job

Try rotation coefficients `0`, `0.05`, `0.15`, and `0.5` on the SAME starting states. Does the cart drift off the track even while the pole stays upright? Find a case where angle-only chooses the opposite action from the predictive controller. Explain it using rotation direction, not just the score.

Optional challenge: add cart position and speed to your controller. Compare on 20 seeded starts. Do not assume a more complicated rule is better until you measure it.


## 6. A continuous state needs a table address

A Q-table stores two estimated action values for each state. But angles such as `0.0401` and `0.0402` would create separate entries if we used raw measurements. Buckets merge nearby observations.

For boundaries `(-0.1, 0, 0.1)`, there are FOUR buckets: below -0.1; from -0.1 to below 0; from 0 to below 0.1; and 0.1 or above. Equality belongs to the upper bucket. `enumerate` yields each boundary's index and value. `len(edges)` is the final bucket number when no earlier return happened.

Velocity has no failure boundary. Our velocity bucket edges are choices about resolution, not physical limits. Each end bucket also includes all more extreme measurements. These coarse choices lose information; Q-maze will give us naturally discrete states instead.

Type this into the empty cell below:

```python
def bucket(value, edges):
    for index, edge in enumerate(edges):
        if value < edge:
            return index
    return len(edges)

EDGES = ((-1.2, 0, 1.2),
         (-0.8, 0, 0.8),
         (-0.12, -0.06, 0, 0.06, 0.12),
         (-1.0, -0.5, 0, 0.5, 1.0))
print(bucket(0.04, EDGES[2]))
```

### Challenge: Address all four measurements

Write `state_key(state)` that returns a tuple of four bucket indices. `zip(state, EDGES)` pairs each measurement with its boundaries. You may use a tuple comprehension or an ordinary loop.

Hint: each paired measurement becomes `bucket(value, edges)`.

Write your answer in the empty cell. Run the following test cell afterward. An assertion failure means the stated requirement has not been met; a NameError usually means you need to run or complete an earlier cell.

In [ ]:
assert state_key((0, 0, 0, 0)) == (2, 2, 3, 3), "Equality goes in the upper bucket."
assert state_key((-9, -9, -9, -9)) == (0, 0, 0, 0)
assert state_key((9, 9, 9, 9)) == (3, 3, 5, 5)
assert state_key((0.01, 0.01, 0.01, 0.01)) == state_key((0.02, 0.02, 0.02, 0.02))
print("Bucket checks passed: nearby states can share a row.")

## 7. Give each action a value

A dictionary maps a key to a value. Here each key is a bucket tuple; each value is a two-item LIST containing left and right action estimates. We use lists because learning changes their entries. New rows start at zero, meaning no estimate has been learned yet—not proof that both actions are worthless.

`if key not in Q` checks whether a row exists. `Q[key]` retrieves it. The returned list is the actual stored row; modifying it modifies the table.

Type this into the empty cell below:

```python
Q = {}
def row(key):
    if key not in Q:
        Q[key] = [0.0, 0.0]
    return Q[key]

print(row(state_key((0, 0, 0.04, 0))))
```

## 8. Calculate a Q-update before programming it

An agent receives reward `1` for a step that stays inside the failure limits and `0` for a failing step. It estimates discounted future reward: how much useful survival remains after this action.

**Target = immediate reward + discount × best next action value.**

**New estimate = old estimate + learning rate × (target − old estimate).**

The learning rate says how far to move toward new evidence. The discount says how much later rewards count. They are different controls. A terminal failure has no future value to include. A training time limit is not physical failure; this notebook still bootstraps at that limit because the underlying task could continue.

Example: old estimate `2`, reward `1`, best next value `4`, discount `0.9`, learning rate `0.25`:

- Target: `1 + 0.9 × 4 = 4.6`.
- Difference: `4.6 − 2 = 2.6`.
- Change: `0.25 × 2.6 = 0.65`.
- New estimate: `2.65`.

Now predict the result if that transition was terminal. The next-state value must disappear. Compute the answer before the test tells you.


### Challenge: Implement the learning calculation

Write `updated_value(old, reward, next_best, terminal, rate=0.25, discount=0.9)` returning the new number. Default arguments supply values when a caller omits them. `max(values)` will later supply the best next action estimate.

Hint: first calculate a target; for terminal transitions the target is just the reward.

Write your answer in the empty cell. Run the following test cell afterward. An assertion failure means the stated requirement has not been met; a NameError usually means you need to run or complete an earlier cell.

In [ ]:
assert math.isclose(updated_value(2, 1, 4, False), 2.65), "Check target, then move a quarter of the difference."
assert math.isclose(updated_value(2, 1, 999, True), 1.75), "Terminal transitions ignore all future value."
assert updated_value(2, 1, 4, False, rate=0) == 2
assert updated_value(2, 1, 4, False, rate=1) == 4.6
print("Q-update checks passed.")

## 9. Sometimes try the other action

Always choosing the current highest estimate can prevent discovery. Epsilon is the chance of deliberately exploring. `random()` produces a number from 0 up to but not including 1. Thus `random() < 0.2` explores roughly 20% of the time.

When estimates tie, choose randomly too: always picking left would introduce a starting bias. `choice` chooses one item from a list. A greedy action maximizes CURRENT estimates; it need not be objectively correct.

Type this into the empty cell below:

```python
def choose_action(key, epsilon, generator):
    values = row(key)
    if generator.random() < epsilon:
        return generator.randrange(2)
    best = max(values)
    candidates = [action for action in (0, 1) if values[action] == best]
    return generator.choice(candidates)
```

In [ ]:
saved = dict(Q)
probe = (99, 99, 99, 99)
Q[probe] = [-2.0, 3.0]
check_rng = Random(11)
assert all(choose_action(probe, 0, check_rng) == 1 for _ in range(30))
assert set(choose_action(probe, 1, check_rng) for _ in range(100)) == {0, 1}
Q.clear()
Q.update(saved)
print("Exploration and greedy choice checks passed.")

## 10. Put the learning loop together

Each attempt starts fresh; the table persists across attempts. For each decision: read the bucket key, choose an action, simulate a transition, check failure, calculate reward, and update ONLY the chosen action's entry.

`next_best` reads a prediction, not a known future outcome. These predictions improve through repeated experience. `scores` records attempted steps. `index + 1` counts attempts from one for reporting. Exploration falls gradually but keeps a 0.05 minimum. `print` reports once per 200 attempts, so it does not flood the browser. Rerunning this cell resumes training on the current table; call `Q.clear()` first for a fresh comparison.

The browser may pause during this bounded synchronous loop. Start with 200 attempts; increase to 1500 after checking speed. There is no promised score threshold.

Type this into the empty cell below:

```python
def train(attempts=200, seed=23, limit=500):
    generator = Random(seed)
    scores = []
    for index in range(attempts):
        current = reset(generator)
        epsilon = max(0.05, 1.0 - index / max(1, attempts * 0.8))
        for tick in range(limit):
            key = state_key(current)
            action = choose_action(key, epsilon, generator)
            following = step(current, action)
            terminal = failed(following)
            reward = 0.0 if terminal else 1.0
            next_best = 0.0 if terminal else max(row(state_key(following)))
            values = row(key)
            values[action] = updated_value(values[action], reward, next_best,
                                           terminal, rate=0.15, discount=0.99)
            current = following
            if terminal:
                break
        scores.append(tick + 1)
        if (index + 1) % 200 == 0:
            print("attempt", index + 1, "recent average", sum(scores[-100:]) / len(scores[-100:]))
    return scores

Q.clear()
scores = train()
print("visited rows:", len(Q))
```

### Discovery: a training curve is not an exam score

Training includes exploratory actions. A low training score can reflect exploration; a high one can reflect a lucky start. Evaluate without deliberate exploration on separate starts. Before more training, record the evaluation below. Then try 1500 attempts from an empty table and compare.

Change ONE thing at a time: bucket edges, learning rate, discount, or exploration schedule. Changing edges invalidates the existing table's meaning; clear it before retraining. Higher resolution means more states to visit, not automatically better learning.


## 11. Compare on the same starts

Evaluation uses a separate generator and fresh starts. Every controller receives the same starts. The learned policy does not update values here; unseen rows are treated as zero-valued without inserting them. The random baseline has its own random generator so its choices do not change the starts or trained policy's tie-breaking sequence.

A mean summarizes all scores. The minimum and maximum show spread. Twenty starts are an initial comparison, not strong statistical evidence. Hand-written control can outperform this small Q-table. That is a useful result.

Type this into the empty cell below:

```python
test_rng = Random(901)
starts = [reset(test_rng) for _ in range(20)]
learned_rng = Random(902)
baseline_rng = Random(903)

def learned_policy(state):
    values = Q.get(state_key(state), [0.0, 0.0])
    best = max(values)
    return learned_rng.choice([a for a in (0, 1) if values[a] == best])

def random_baseline(state):
    return baseline_rng.randrange(2)

for policy in (random_baseline, angle_policy, predictive_policy, learned_policy):
    results = [episode(policy, start)[0] for start in starts]
    print(policy.__name__, "mean:", round(sum(results) / len(results), 1),
          "min:", min(results), "max:", max(results))
```

### Challenge: Measure success without hiding failures

Write `success_fraction(results, limit=500)`: return the fraction of scores at least as large as `limit`. For an empty list return `0.0`. You may count with a loop. Do not use the average score as a substitute: one excellent attempt can hide several failures.

Write your answer in the empty cell. Run the following test cell afterward. An assertion failure means the stated requirement has not been met; a NameError usually means you need to run or complete an earlier cell.

In [ ]:
assert success_fraction([]) == 0.0
assert success_fraction([500, 10, 500, 20]) == 0.5
assert success_fraction([50, 49], limit=50) == 0.5
assert success_fraction([501], limit=500) == 1.0
print("Success metric checks passed.")

## 12. Explain what you built

Answer in your own words in the Markdown cell below:

1. Why can two equal pole angles need different actions?
2. What does one Q-table row represent? What do its two entries estimate?
3. Why do buckets discard information? Give two different states that share a row.
4. Calculate an update with old value 3, reward 1, next best 5, rate 0.2, discount 0.9. Repeat for terminal failure with reward 0.
5. Where in `train` does experience change future decisions?
6. Why must terminal failure omit the next value, while a time limit need not?
7. What evidence would justify saying the agent learned, rather than got lucky?
8. Which parts are simulated physics, which are hand-written control, and which are learning?

**Next: Q-maze.** Keep the same ideas—state, action, reward, episode, table, exploration, update—but replace four continuous measurements with a grid cell. You will be able to inspect every table row and trace how useful estimates propagate backward from a goal.


Write your explanation here.

## Optional: a browser picture of one state

The required notebook needs no graphics library. If your Pyodide notebook supports `IPython.display`, the following OPTIONAL cell can draw a state as SVG. Skip it if imports fail. `SVG` wraps XML text for the notebook host; `display` requests rich output. An f-string substitutes computed coordinates inside `{...}`. SVG uses pixels and its vertical coordinate increases downward, so we subtract the pole's upward height.

This is a static picture, not an animation or a required dependency.

```python
from IPython.display import SVG, display

def picture(state):
    x, velocity, angle, omega = state
    cx = 300 + x * 90
    tip_x = cx + 100 * math.sin(angle)
    tip_y = 160 - 100 * math.cos(angle)
    svg = f'<svg xmlns="http://www.w3.org/2000/svg" width="600" height="220"><line x1="20" y1="175" x2="580" y2="175" stroke="gray"/><rect x="{cx-20}" y="160" width="40" height="20" fill="steelblue"/><line x1="{cx}" y1="160" x2="{tip_x}" y2="{tip_y}" stroke="orange" stroke-width="8"/></svg>'
    display(SVG(svg))

picture(history[0])
```


# Hints and worked challenge solutions

Stop here until you have tried the challenges. These solve the programming tasks, not the discovery questions. Type a solution only after explaining why it works. Then rerun its tests.


### Recognize failure

```python
def failed(state):
    x, velocity, angle, angular_velocity = state
    return abs(x) > 2.4 or abs(angle) > 12 * math.pi / 180
```

### Write an angle-only controller

```python
def angle_policy(state):
    return 1 if state[2] > 0 else 0
```

### Address all four measurements

```python
def state_key(state):
    return tuple(bucket(value, edges) for value, edges in zip(state, EDGES))
```

### Implement the learning calculation

```python
def updated_value(old, reward, next_best, terminal, rate=0.25, discount=0.9):
    target = reward if terminal else reward + discount * next_best
    return old + rate * (target - old)
```

### Measure success without hiding failures

```python
def success_fraction(results, limit=500):
    if not results:
        return 0.0
    successes = sum(1 for score in results if score >= limit)
    return successes / len(results)
```